# Lekcja 11 - Protokół Kontekstu Modelu (MCP)

The **Protokół Kontekstu Modelu (MCP)** is an open standard that enables agents to dynamically discover and use tools, resources, and data sources at runtime. Instead of hardcoding tools into an agent, MCP lets agents connect to external servers that expose capabilities on demand.

In this lesson, you'll learn:
- Czym jest MCP i dlaczego ma znaczenie dla systemów agentowych
- Jak działa architektura klient-serwer MCP
- Jak zbudować agentów, którzy używają wykrywania narzędzi w stylu MCP


## Konfiguracja

**Wymagania wstępne:**
- Projekt Azure AI Foundry z wdrożonym modelem
- Uruchom `az login` w celu uwierzytelnienia


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Czym jest Model Context Protocol (MCP)?

MCP definiuje standardowy sposób, w jaki agenci AI mogą odkrywać i wchodzić w interakcje z zewnętrznymi narzędziami i źródłami danych:

- **MCP Server**: Udostępnia narzędzia, zasoby i prompty za pomocą standardowego protokołu
- **MCP Client**: Środowisko uruchomieniowe agenta, które łączy się z serwerami i wykrywa dostępne możliwości
- **Dynamic Discovery**: Agenci nie potrzebują na sztywno zapisanych narzędzi — odkrywają, co jest dostępne w czasie działania

To potężne narzędzie do budowania rozszerzalnych systemów agentów, w których nowe możliwości można dodawać bez modyfikowania kodu agenta.


## Jak działa MCP

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. Agent (klient MCP) łączy się z serwerem MCP
2. Serwer zwraca listę dostępnych narzędzi i ich schematów
3. Agent może następnie wywołać dowolne odkryte narzędzie podczas swojego rozumowania
4. Wyniki przepływają z powrotem tym samym protokołem


## Symulacja wykrywania narzędzi MCP

Ponieważ prawdziwy serwer MCP wymaga uruchomionego procesu serwera, zademonstrujemy ten wzorzec, używając funkcji `@tool`, które symulują to, co dostarczyłby serwis zakwaterowania połączony z MCP.

W środowisku produkcyjnym te narzędzia byłyby odkrywane dynamicznie z serwera MCP, zamiast być definiowane lokalnie.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## Tworzenie agenta z narzędziami w stylu MCP


In [ ]:
agent = await provider.create_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP w środowisku produkcyjnym

W środowisku produkcyjnym MCP umożliwia zaawansowane wzorce:

- **Dynamiczne wykrywanie narzędzi**: Agenci łączą się z serwerami MCP i odkrywają narzędzia w czasie wykonywania
- **Architektura rozdzielona**: Dostawcy narzędzi mogą wprowadzać aktualizacje niezależnie od agenta
- **Udostępnianie między organizacjami**: Zespoły mogą udostępniać możliwości za pośrednictwem serwerów MCP, z których może korzystać dowolny agent
- **Wsparcie dla Microsoft Agent Framework**: MAF ma wbudowane wsparcie klienta MCP poprzez integrację `mcp`

Aby użyć rzeczywistego serwera MCP z MAF, połączysz się za pomocą `hosted_mcp_tool()` lub integracji klienta MCP.

**Dowiedz się więcej:**
- [Specyfikacja MCP](https://modelcontextprotocol.io/)
- [Obsługa MCP w Microsoft Agent Framework](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## Podsumowanie

W tej lekcji dowiedziałeś się:
- **MCP** jest otwartym standardem do dynamicznego wykrywania narzędzi pomiędzy agentami a dostawcami narzędzi
- **architektura klient-serwer** pozwala agentom odkrywać możliwości w czasie wykonywania
- MCP umożliwia **rozszerzalne, luźno powiązane systemy agentów**, gdzie narzędzia można dodawać bez zmian w kodzie
- Microsoft Agent Framework zapewnia **wbudowane wsparcie MCP** do użytku produkcyjnego


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Zastrzeżenie:
Ten dokument został przetłumaczony przy użyciu usługi tłumaczeń opartych na sztucznej inteligencji [Co-op Translator](https://github.com/Azure/co-op-translator). Chociaż dokładamy starań, aby tłumaczenia były jak najdokładniejsze, prosimy pamiętać, że automatyczne tłumaczenia mogą zawierać błędy lub nieścisłości. Oryginalny dokument w języku źródłowym należy traktować jako źródło autorytatywne. W przypadku istotnych informacji zalecane jest skorzystanie z profesjonalnego tłumaczenia wykonanego przez człowieka. Nie ponosimy odpowiedzialności za jakiekolwiek nieporozumienia lub błędne interpretacje wynikające z korzystania z tego tłumaczenia.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
